# Mnemos held-out evaluation

This runs the frozen held-out corpus against every evaluation profile at once on one GPU. Ollama serves Qwen3.5 9B and the Nomic embedding model locally, and each profile runs as its own process against it, instead of one after another on a single consumer card.

**Before you start**

1. On the machine with the repo and the Ollama models, build a bundle:
   `python -m benchmarks.local_memory.prepare_colab_bundle --corpus <corpus.jsonl> --output <folder>`
2. Upload that folder to Google Drive under `MyDrive/mnemos-eval/`.
3. Pick an A100 or H100 runtime (Runtime -> Change runtime type), then set `BUNDLE_NAME` below and run the cells in order. A bundle built with `--judge typesafe` also needs your TypeSafe key in Colab's Secrets panel as `TYPESAFE_API_KEY`; nothing else changes, and the GPU only serves embeddings (and the local model if surprisal is in prediction mode).

**If the runtime disconnects**, reconnect and run everything again. Finished profiles are skipped, and the rest resume from the checkpoints mirrored to Drive every five minutes. A resume on a different GPU model is refused on purpose, since mixing hardware inside one profile would blur the comparison.

Everything lands in `MyDrive/mnemos-eval/<bundle>/run/`. The file to send back is `summary.json`, along with `reports/` if you want the raw observations.

In [ ]:
# Folder name of the uploaded bundle under DRIVE_ROOT.
BUNDLE_NAME = "mnemos-heldout-CHANGE-ME"
DRIVE_ROOT = "/content/drive/MyDrive/mnemos-eval"
# None runs every profile in the bundle, or name a subset such as ["baseline", "full"].
PROFILES = None
# Requests Ollama serves at once per model. Seven profiles fit comfortably in 8.
NUM_PARALLEL = 8

In [ ]:
import json
import os
import pathlib
import subprocess
import sys
import time
import urllib.request

from google.colab import drive

drive.mount("/content/drive")
BUNDLE = pathlib.Path(DRIVE_ROOT) / BUNDLE_NAME
MIRROR = BUNDLE / "run"  # durable copy on Drive
LOCAL = pathlib.Path("/content/mnemos-run")  # live work on the VM's disk
SOURCE = pathlib.Path("/content/mnemos-src")
PREP = json.loads((BUNDLE / "preparation.json").read_text())
PROFILE_LIST = PROFILES or PREP["profiles"]
ENDPOINT = "http://127.0.0.1:11434"
print(json.dumps({key: PREP[key] for key in (
    "source_commit", "dataset_version", "heldout_sessions", "repetitions",
    "model", "embedding_model", "ollama_version", "ollama_context_length",
)}, indent=2))
print("profiles:", PROFILE_LIST)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## Ollama

Pinned to the version and context length the bundle recorded, so the only intended difference from the local run is the GPU.

In [ ]:
OLLAMA_VERSION = PREP["ollama_version"]
installed = subprocess.run(["bash", "-lc", "ollama --version 2>/dev/null"], capture_output=True, text=True).stdout
if OLLAMA_VERSION not in installed:
    # Recent Linux builds ship as .tar.zst, which the install script needs zstd to unpack.
    !apt-get -qq update && apt-get -qq install -y zstd pciutils > /dev/null
    !curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION={OLLAMA_VERSION} sh

SERVER_ENV = dict(
    os.environ,
    OLLAMA_CONTEXT_LENGTH=str(PREP["ollama_context_length"]),
    OLLAMA_NUM_PARALLEL=str(NUM_PARALLEL),
    OLLAMA_MAX_LOADED_MODELS="2",
    OLLAMA_KEEP_ALIVE="24h",
    OLLAMA_MODELS="/content/ollama-models",
)


def server_up():
    try:
        with urllib.request.urlopen(ENDPOINT + "/api/version", timeout=2) as response:
            return json.loads(response.read())["version"]
    except OSError:
        return None


if server_up() is None:
    subprocess.Popen(
        ["ollama", "serve"], env=SERVER_ENV,
        stdout=open("/content/ollama.log", "a"), stderr=subprocess.STDOUT,
    )
    for _ in range(120):
        if server_up():
            break
        time.sleep(1)
print("ollama", server_up())

In [ ]:
for name in (PREP["model"], PREP["embedding_model"]):
    subprocess.run(["ollama", "pull", name], check=True, env=SERVER_ENV)

with urllib.request.urlopen(ENDPOINT + "/api/tags") as response:
    live = {item["name"]: item["digest"] for item in json.loads(response.read())["models"]}
for name, record in PREP["models"].items():
    same = live.get(name) == record["digest"]
    print(("ok      " if same else "MISMATCH"), name, record["digest"][:12], str(live.get(name))[:12])
assert all(live.get(name) == record["digest"] for name, record in PREP["models"].items()), (
    "A model on Colab differs from the one the bundle recorded. Stop here rather than compare different weights."
)

## Source

Unpacks the exact commit from the bundle and checks every file against the recorded digests. The first session writes the installed dependency versions to Drive, and later sessions install those same versions, because they are part of each checkpoint's identity.

In [ ]:
import shutil
import zipfile
from importlib import metadata

if SOURCE.exists():
    shutil.rmtree(SOURCE)
zipfile.ZipFile(BUNDLE / "source.zip").extractall(SOURCE)

pins = MIRROR / "pinned-requirements.txt"
requirements = pins.read_text().split() if pins.exists() else PREP["python_dependencies"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *requirements], check=True)
if not pins.exists():
    MIRROR.mkdir(parents=True, exist_ok=True)
    names = ["numpy", "pydantic", "pydantic-core", "httpx", "sqlite-vec", "platformdirs",
             "fastembed", "tomlkit", "pyyaml"]
    pins.write_text("\n".join(f"{name}=={metadata.version(name)}" for name in names) + "\n")
print(pins.read_text())

sys.path.insert(0, str(SOURCE))
from benchmarks.local_memory.colab_heldout import aggregate, verify_bundle

verify_bundle(BUNDLE, SOURCE)
print("bundle verified at", PREP["source_commit"])

## Run

Starts one process per profile. Re-running this cell is safe: finished profiles are skipped and live ones are left alone.

In [ ]:
for folder in ("work", "reports", "status", "logs"):
    (LOCAL / folder).mkdir(parents=True, exist_ok=True)
RUN_ENV = dict(
    os.environ,
    OLLAMA_CONTEXT_LENGTH=str(PREP["ollama_context_length"]),
    OLLAMA_NUM_PARALLEL=str(NUM_PARALLEL),
    MNEMOS_AUTO_START="0",
    PYTHONUNBUFFERED="1",
)
if PREP["options"].get("judge_provider") == "typesafe":
    # The key lives in Colab's Secrets panel (key icon on the left) as TYPESAFE_API_KEY.
    from google.colab import userdata

    RUN_ENV["MNEMOS_TYPESAFE_API_KEY"] = userdata.get("TYPESAFE_API_KEY")
    print("judged run on TypeSafe:", PREP["options"].get("typesafe_model"),
          "surprisal:", PREP["options"].get("surprisal_mode"))
FINISHED = {"completed", "incomplete"}


def mirrored_status(name):
    path = MIRROR / "status" / f"{name}.json"
    return json.loads(path.read_text()) if path.exists() else {}


def runner_alive(name):
    # Runners can outlive a kernel restart, so look for the process itself.
    pattern = f"colab_heldout run .*--profile {name}( |$)"
    return subprocess.run(["pgrep", "-f", pattern], capture_output=True).returncode == 0


for name in PROFILE_LIST:
    if mirrored_status(name).get("status") in FINISHED:
        print(f"{name}: already finished")
        continue
    if runner_alive(name):
        print(f"{name}: still running")
        continue
    log = open(LOCAL / "logs" / f"{name}.log", "a")
    process = subprocess.Popen(
        [sys.executable, "-m", "benchmarks.local_memory.colab_heldout", "run",
         "--bundle", str(BUNDLE), "--source", str(SOURCE), "--run", str(LOCAL),
         "--mirror", str(MIRROR), "--profile", name, "--endpoint", ENDPOINT],
        cwd=SOURCE, env=RUN_ENV, stdout=log, stderr=subprocess.STDOUT,
    )
    print(f"{name}: started, pid {process.pid}")

In [ ]:
def completed(name):
    checkpoint = LOCAL / "work" / f"{name}.checkpoint.json"
    try:
        return len(json.loads(checkpoint.read_text())["completed_observation_ids"])
    except (OSError, ValueError, KeyError):
        return mirrored_status(name).get("completed_observations")


def show():
    expected = PREP["heldout_sessions"] * PREP["repetitions"]
    for name in PROFILE_LIST:
        state = "running" if runner_alive(name) else mirrored_status(name).get("status", "not started")
        print(f"{name:32} {str(completed(name)):>5} / {expected}   {state}")


# Keeps a cell busy so the runtime counts as active. Interrupt it any time;
# the profiles keep running.
while any(runner_alive(name) for name in PROFILE_LIST):
    show()
    print(time.strftime("%H:%M:%S"), "- next update in 10 minutes\n")
    time.sleep(600)
show()

A profile that shows `failed` has its log in `/content/mnemos-run/logs/<profile>.log`, mirrored to Drive. Fix the cause, then re-run the Run cell; it resumes from the last checkpoint.

## Results

Pairs every finished profile with the baseline, writes the comparisons into `reports/baseline.json`, and saves `summary.json` next to them. Each line is the paired difference from the baseline with its 95% bootstrap interval, clustered by session.

In [ ]:
summary = aggregate(mirror=MIRROR, repetitions=PREP["repetitions"])
print("statuses:", summary["statuses"])
if summary["missing_profiles"]:
    print("not finished yet:", summary["missing_profiles"])
for name, comparison in summary["paired_comparisons"].items():
    for metric, value in comparison["metrics"].items():
        if "estimate" in value:
            print(f"{name:32} {metric:22} {value['estimate']:+.3f} "
                  f"[{value['lower']:+.3f}, {value['upper']:+.3f}]  n={value['sample_count']}")
        else:
            print(f"{name:32} {metric:22} not measured")
print("\nsummary written to", MIRROR / "summary.json")